In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd

In [ ]:
PATH = "output-script/timeseries.parquet"

UPSTREAM_NAME = "upstream control zone"
DOWNSTREAM_NAME = "downstream influence zone"

In [ ]:
# https://forum.dataspace.copernicus.eu/t/inconsistent-band-names-csv-parquet/5683
# ⚠️ openEO Parquet exports "band_0", etc.
# map these back to the CSV bands.
BAND_RENAME = {
    "band_0": "sum(hotspot)",
    "band_1": "count(hotspot)",
    "band_2": "avg(hotspot)",
    "band_3": "sum(hotspot_exceedance_count)",
    "band_4": "count(hotspot_exceedance_count)",
    "band_5": "avg(hotspot_exceedance_count)",
    "band_6": "sum(water_valid)",
    "band_7": "count(water_valid)",
    "band_8": "avg(water_valid)",
}

In [ ]:
gdf = gpd.read_parquet(PATH)
gdf = gdf.rename(columns=BAND_RENAME)

In [ ]:
# these are the data that we need for WQRI
gdf["N_hotspot"] = gdf["sum(hotspot)"]
gdf["C_bar"] = gdf["avg(hotspot_exceedance_count)"]
gdf["N_valid"] = gdf["sum(water_valid)"]
gdf["year"] = pd.to_datetime(gdf["date"], utc=True).dt.year

In [ ]:
upstream = gdf.loc[gdf["name"] == UPSTREAM_NAME]
upstream = upstream[["year", "N_hotspot", "C_bar", "N_valid"]]
upstream = upstream.set_index("year")
upstream.head()

In [ ]:
downstream = gdf.loc[gdf["name"] == DOWNSTREAM_NAME]
downstream = downstream[["year", "N_hotspot", "C_bar", "N_valid"]]
downstream = downstream.set_index("year")
downstream.head()

In [ ]:
def hfi(n_hotspot: float, n_valid: float) -> float:
    """Hotspot Frequency Index (HFI)

    Fraction of valid water pixels classified as hotspots in a zone for one year.
    """
    if n_valid == 0:
        return np.nan
    return n_hotspot / n_valid


def pdi_star(mean_exceedance: float) -> float:
    """Normalised Persistence Index (PDI*)

    Mean annual exceedance count among hotspot pixels, scaled by the maximum
    possible anomalous months (12).
    """
    return mean_exceedance / 12.0


def sei(hfi_d: float, hfi_u: float) -> float:
    """Spatial Exposure Index (SEI)

    Downstream hotspot frequency relative to the combined upstream and
    downstream frequency

    Values above 0.5 indicate higher hotspot frequency downstream than upstream.

    Args:
        hfi_d: Hotspot Frequency Index in the downstream influence zone.
        hfi_u: Hotspot Frequency Index in the upstream control zone.
    """
    denominator = hfi_d + hfi_u
    if denominator == 0:
        return np.nan
    return hfi_d / denominator

In [ ]:
hfi_u = upstream.apply(
    lambda row: hfi(row["N_hotspot"], row["N_valid"]),
    axis=1,
)
pdi_star_u = upstream["C_bar"].apply(pdi_star)

In [ ]:
hfi_d = downstream.apply(
    lambda row: hfi(row["N_hotspot"], row["N_valid"]),
    axis=1,
)
pdi_star_d = downstream["C_bar"].apply(pdi_star)

In [ ]:
sei = hfi_d.combine(hfi_u, sei)

In [ ]:
wqri_df = pd.DataFrame(
    {
        "HFI_U": hfi_u,
        "PDI*_U": pdi_star_u,
        "HFI_D": hfi_d,
        "PDI*_D": pdi_star_d,
        "SEI": sei,
    }
)

wqri_df.head()

In [ ]:
# ⚠️ I'm assuming that the PDI* input is for the downstream influence zone.
# but this is not explicitly stated in the ATBD

def wqri(
    hfi_d: float,
    pdi_star_d: float,
    sei: float,
    *,
    w_h: float = 1 / 3,
    w_p: float = 1 / 3,
    w_s: float = 1 / 3,
) -> float:
    """Water Quality Risk Index (WQRI)

    Arguments:
        hfi_d: Hotspot Frequency Index (HFI) for downstream influence zone
        pdi_star_d: Normalised Persistence Index (PDI*) for the downstream influence zone.
        sei: Spatial Exposure Index
        w_h: Weight on HFI.
        w_p: Weight on PDI*.
        w_s: Weight on SEI.
    """
    return w_h * hfi_d + w_p * pdi_star_d + w_s * sei

In [ ]:
wqri_df["WQRI"] = wqri_df.apply(
    lambda row: wqri(row["HFI_D"], row["PDI*_D"], row["SEI"]),
    axis=1,
)
wqri_df